# RouteRanker — Module 1: Data Collection and Dataset Understanding

This recreated notebook inspects the supplied GTFS CSV/TXT ZIP without changing raw records. **Run all cells in Google Colab before submitting.** Outputs are intentionally empty until execution; no results or collection history are fabricated.

Scope: inventory every feed folder, inspect structure and samples, count missing values and duplicates, validate identifiers, times, dates and coordinates, and generate reports. This is an initial inspection, not complete GTFS certification. No cleaning or machine learning is performed.

## 1. Source documentation
The assignment names Lepton Software: https://datarade.ai/data-products/india-transit-data-railways-metro-and-public-buses-routes-lepton-software

The supplied archive is user-provided. Its original download URL, download date, licence, city coverage and relationship to Lepton **must be verified manually**. Publisher metadata is evidence from the feed, not proof of where the archive was obtained. Do not claim 23 cities collected. Folder names are feed labels, not verified cities.

CSV exports are accepted for inspection; preserve the original GTFS archives separately. Complete these fields with evidence before submission.

In [ ]:
SOURCE_NOTES = {
    "supplied_archive_origin": "User-provided; original source not yet verified",
    "original_download_url": "Not recorded",
    "original_download_date": "Not recorded",
    "licence": "Not verified",
    "lepton_access_status": "Not checked in this notebook",
    "verified_city_coverage": "Requires review of agency, stops and source documentation"
}

## 2. Upload and safely extract the ZIP
Upload All_GTFS_CSV_Files.zip or its renamed copy. All source tables remain unchanged.

In [ ]:
from google.colab import files
from pathlib import Path
from IPython.display import display
import pandas as pd
import numpy as np
import zipfile, hashlib, json, shutil, gc

uploaded = files.upload()
archives = [Path(x) for x in uploaded if x.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one GTFS ZIP.')
PROJECT = Path('/content/RouteRanker_Module1')
RAW = PROJECT / 'data/raw'
REPORTS = PROJECT / 'reports'
RAW.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
# A new extraction directory prevents mixing files from different uploads.
archive_path = archives[0]
archive_hash = hashlib.sha256(archive_path.read_bytes()).hexdigest()
EXTRACT = RAW / archive_hash[:12]
EXTRACT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_path) as z:
    root = EXTRACT.resolve()
    for member in z.infolist():
        target = (EXTRACT / member.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError('Unsafe ZIP member path.')
    z.extractall(EXTRACT)
paths = sorted(p for p in EXTRACT.rglob('*') if p.suffix.lower() in ['.csv', '.txt'] and '__MACOSX' not in p.parts)
if not paths:
    raise ValueError('No CSV/TXT tables found; check archive contents.')
feeds = {}
for p in paths:
    feeds.setdefault(p.parent, []).append(p)
print('Feed folders:', len(feeds))
print('Tables:', len(paths))
print('Archive SHA256:', archive_hash)

## 3. Inventory, types and samples
Tables are processed one at a time to limit memory. IDs and source values are read as strings; numeric validity is checked explicitly later. Empty and whitespace-only cells are counted as missing observations. Duplicate detection below compares complete original rows. Optional fields may legitimately be empty.

In [ ]:
inventory, columns_report, samples, failures = [], [], [], []
for folder, feed_paths in feeds.items():
    label = str(folder.relative_to(EXTRACT))
    for p in feed_paths:
        print('Inspecting:', label, p.name, flush=True)
        try:
            df = pd.read_csv(p, dtype='string', keep_default_na=False, encoding='utf-8-sig', low_memory=False)
            missing = df.apply(lambda c: c.str.strip().eq('')).sum()
            inventory.append({'feed_folder': label, 'file': p.name, 'records': len(df), 'column_count': len(df.columns), 'columns': ', '.join(df.columns), 'empty_cells': int(missing.sum()), 'duplicate_rows': int(df.duplicated().sum()), 'status': 'Loaded'})
            for col in df:
                columns_report.append({'feed_folder': label, 'file': p.name, 'column': col, 'loaded_dtype': str(df[col].dtype), 'empty_values': int(missing[col]), 'sample_values': json.dumps(df[col].head(3).tolist(), ensure_ascii=False)})
            samples.append({'feed_folder': label, 'file': p.name, 'sample_rows': df.head(3).to_dict('records')})
            del df
            gc.collect()
        except Exception as e:
            failures.append({'feed_folder': label, 'file': p.name, 'error': str(e)})
            inventory.append({'feed_folder': label, 'file': p.name, 'status': 'FAILED', 'error': str(e)})
inventory_df = pd.DataFrame(inventory)
column_df = pd.DataFrame(columns_report)
display(inventory_df)
display(column_df.head(20))

## 4. Availability, relationships and initial validation
Relationships: routes.route_id → trips.route_id; trips.trip_id → stop_times.trip_id; stops.stop_id → stop_times.stop_id; the union of calendar and calendar_dates service IDs → trips.service_id. Optional shapes and frequencies support geometry and frequency-based service inspection.

Calendar.txt can legitimately be absent when calendar_dates defines service dates. Blank times may be permitted for some intermediate stops. Hours above 23 are permitted; malformed minutes and seconds are not. A finding is a review item, not an instruction to delete records.

In [ ]:
availability, checks, metadata = [], [], []
required_cols = {
 'routes': ['route_id', 'route_type'],
 'trips': ['route_id', 'service_id', 'trip_id'],
 'stops': ['stop_id'],
 'stop_times': ['trip_id', 'stop_id', 'stop_sequence'],
 'calendar': ['service_id', 'start_date', 'end_date', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'],
 'calendar_dates': ['service_id', 'date', 'exception_type']
}
for folder, feed_paths in feeds.items():
    label = str(folder.relative_to(EXTRACT))
    tables = {}
    def add(check, count, interpretation):
        checks.append({'feed_folder': label, 'check': check, 'affected_records': int(count), 'interpretation': interpretation})
    # Load main inspection tables only; shapes are inventoried without retaining large geometry.
    for p in feed_paths:
        name = p.stem.lower()
        if name not in set(required_cols) | {'agency', 'feed_info', 'frequencies'}:
            continue
        if name in tables:
            add('Multiple files for table: ' + name, 1, 'Ambiguous CSV/TXT copies; first file used for checks.')
            continue
        try:
            tables[name] = pd.read_csv(p, dtype='string', keep_default_na=False, encoding='utf-8-sig', low_memory=False)
        except Exception as e:
            add('Load failure: ' + name, 1, str(e))
    names = {p.stem.lower() for p in feed_paths}
    availability.append({'feed_folder': label, 'missing_main_tables': ', '.join(sorted({'routes','trips','stop_times','stops'} - names)) or 'None', 'calendar': 'calendar' in names, 'calendar_dates': 'calendar_dates' in names, 'shapes': 'shapes' in names, 'frequencies': 'frequencies' in names})
    if not {'calendar','calendar_dates'} & names:
        add('No operating calendar table', 1, 'Operating dates cannot be established.')
    for name, cols in required_cols.items():
        if name in tables:
            for col in cols:
                if col not in tables[name]:
                    add(f'{name}: missing column {col}', 1, 'Required for this inspection calculation; investigate source schema.')
    for name, key in [('routes','route_id'), ('trips','trip_id'), ('stops','stop_id'), ('calendar','service_id')]:
        if name in tables and key in tables[name]:
            s = tables[name][key]
            add(f'{name}: blank {key}', s.str.strip().eq('').sum(), 'Identifier requires investigation.')
            add(f'{name}: repeated {key}', s.duplicated(keep=False).sum(), 'Count includes all rows participating in repeated keys.')
    for child, key, parent in [('trips','route_id','routes'), ('stop_times','trip_id','trips'), ('stop_times','stop_id','stops'), ('frequencies','trip_id','trips')]:
        if child in tables and parent in tables and key in tables[child] and key in tables[parent]:
            s = tables[child][key]
            add(f'{child}.{key}: unmatched reference', (s.str.strip().ne('') & ~s.isin(tables[parent][key])).sum(), 'Nonblank reference absent from parent table.')
    services = set()
    for name in ['calendar','calendar_dates']:
        if name in tables and 'service_id' in tables[name]:
            services.update(tables[name]['service_id'])
    if 'trips' in tables and 'service_id' in tables['trips']:
        s = tables['trips']['service_id']
        add('trips: blank service_id', s.str.strip().eq('').sum(), 'Missing service identifier.')
        add('trips: unmatched service_id', (s.str.strip().ne('') & ~s.isin(services)).sum(), 'Compared against both calendar tables.')
    if 'stops' in tables:
        for col, lo, hi in [('stop_lat',-90,90), ('stop_lon',-180,180)]:
            if col in tables['stops']:
                s = tables['stops'][col]; v = pd.to_numeric(s, errors='coerce')
                add('stops: blank ' + col, s.str.strip().eq('').sum(), 'Requiredness depends on stop location_type; review.')
                add('stops: invalid nonblank ' + col, (s.str.strip().ne('') & (v.isna() | ~v.between(lo, hi))).sum(), 'Numeric/range check, not geographic completeness validation.')
    for name in ['calendar','calendar_dates']:
        if name not in tables: continue
        df = tables[name]; dates = {}
        for col in ['start_date','end_date','date']:
            if col in df:
                dates[col] = pd.to_datetime(df[col].where(df[col].str.fullmatch(r'\d{8}')), format='%Y%m%d', errors='coerce')
                add(f'{name}: invalid {col}', dates[col].isna().sum(), 'Expected a real YYYYMMDD date.')
        if 'start_date' in dates and 'end_date' in dates:
            add('calendar: start after end', (dates['start_date'] > dates['end_date']).sum(), 'Invalid date range.')
            today = pd.Timestamp.now(tz='Asia/Kolkata').tz_localize(None).normalize()
            add('calendar: ended before inspection date', dates['end_date'].lt(today).sum(), 'Historical calendar; not necessarily an error.')
        for col in ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']:
            if col in df: add('calendar: invalid ' + col, (~df[col].isin(['0','1'])).sum(), 'Weekday flags must be 0 or 1.')
        if 'exception_type' in df:
            add('calendar_dates: invalid exception_type', (~df['exception_type'].isin(['1','2'])).sum(), 'Expected 1 addition or 2 removal.')
        if {'service_id','date'}.issubset(df.columns):
            add('calendar_dates: repeated key', df.duplicated(['service_id','date'], keep=False).sum(), 'Review duplicate service/date pairs.')
    if 'stop_times' in tables:
        df = tables['stop_times']; seconds = {}
        for col in ['arrival_time','departure_time']:
            if col not in df: continue
            s = df[col]; blank = s.str.strip().eq(''); valid = s.str.fullmatch(r'\d{2,}:[0-5]\d:[0-5]\d')
            add('stop_times: blank ' + col, blank.sum(), 'May be allowed at intermediate stops; inspect context.')
            add('stop_times: malformed ' + col, (~blank & ~valid).sum(), 'Hours >=24 accepted; minutes/seconds restricted to 00–59.')
            parts = s.where(valid).str.extract(r'^(\d+):(\d{2}):(\d{2})$').apply(pd.to_numeric, errors='coerce')
            seconds[col] = parts[0]*3600 + parts[1]*60 + parts[2]
        if len(seconds) == 2:
            add('stop_times: departure before arrival', (seconds['departure_time'] < seconds['arrival_time']).sum(), 'Compare within same stop visit.')
        if 'stop_sequence' in df:
            seq = pd.to_numeric(df['stop_sequence'], errors='coerce')
            bad = seq.isna() | seq.lt(0) | seq.mod(1).ne(0)
            add('stop_times: invalid stop_sequence', bad.sum(), 'Expected nonnegative integer.')
            if 'trip_id' in df:
                tmp = pd.DataFrame({'trip_id':df['trip_id'], 'sequence':seq})
                add('stop_times: repeated trip/sequence', tmp.duplicated(['trip_id','sequence'],keep=False).sum(), 'Repeated numeric sequences within trip.')
                if len(seconds) == 2:
                    tmp['arrival'] = seconds['arrival_time']; tmp['departure'] = seconds['departure_time']
                    ordered = tmp.loc[~bad].sort_values(['trip_id','sequence'])
                    previous = ordered.groupby('trip_id')['departure'].shift()
                    add('stop_times: arrival before previous departure', (ordered['arrival'] < previous).sum(), 'Review time progression between adjacent stop visits.')
    for name in ['agency','feed_info']:
        if name in tables:
            metadata.append({'feed_folder':label, 'table':name, 'records':tables[name].to_dict('records')})
    del tables
    gc.collect()
availability_df = pd.DataFrame(availability)
quality_df = pd.DataFrame(checks)
display(availability_df)
display(quality_df[quality_df['affected_records'] > 0])

## 5. Interpret the results
Write your own observations below after running the notebook:

- Verified city/operator per folder, with evidence:
- Files successfully loaded and missing tables:
- Counts and important columns:
- Empty optional fields versus required-field concerns:
- Duplicates and unmatched references:
- Historical calendars and unusual times:
- Additional checks required:

Optional shapes support maps and geometry estimates. Frequencies describe headway-based operations; trip templates are not automatically individual departures. Scheduled GTFS data does not establish ridership, occupancy, actual traffic or underutilisation.

## 6. Save inventory and reports
Generated reports reflect this execution. Complete source verification and written interpretation before submitting.

In [ ]:
inventory_df.to_csv(REPORTS / 'dataset_inventory.csv', index=False)
column_df.to_csv(REPORTS / 'column_inventory.csv', index=False)
availability_df.to_csv(REPORTS / 'file_availability.csv', index=False)
quality_df.to_csv(REPORTS / 'data_quality_observations.csv', index=False)
(REPORTS / 'sample_rows.json').write_text(json.dumps(samples, ensure_ascii=False, indent=2), encoding='utf-8')
(REPORTS / 'feed_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
source = dict(SOURCE_NOTES, uploaded_archive=archive_path.name, sha256=archive_hash)
(REPORTS / 'source_documentation.json').write_text(json.dumps(source, indent=2), encoding='utf-8')
relationships = """# GTFS relationships
- routes.route_id -> trips.route_id: route to its trips.
- trips.trip_id -> stop_times.trip_id: trip to ordered stop visits.
- stops.stop_id -> stop_times.stop_id: stop location and description.
- calendar/calendar_dates.service_id -> trips.service_id: operating dates.
- shapes.shape_id -> trips.shape_id, when provided: path geometry.
- trips.trip_id -> frequencies.trip_id, when provided: frequency rules.

IDs are labels, not numerical measurements. Calendar exceptions can add/remove service.
Missing calendar.txt is not automatically invalid if calendar_dates.txt defines dates.
"""
(REPORTS / 'relationships.md').write_text(relationships, encoding='utf-8')
summary = ['# Module 1 dataset inventory', '', f'Feed folders inspected: {len(feeds)}', f'Table files discovered: {len(paths)}', f'Load failures: {len(failures)}', '', '## Feed folders (city coverage requires verification)']
for row in availability:
    summary.append('- ' + row['feed_folder'] + ': missing main tables = ' + row['missing_main_tables'])
summary += ['', '## Interpretation limits', 'Raw values were not modified. Review optional-field missingness, historical calendars and source provenance. This is not complete GTFS validation. No passenger data or machine learning results are claimed.', '', '## Written observations', 'Complete the interpretation section in the notebook before submission.']
(REPORTS / 'module1_dataset_inventory.md').write_text('\n'.join(summary), encoding='utf-8')
print('Reports saved:', REPORTS)
print('Load failures:', len(failures))
print('Review findings before declaring Module 1 complete.')

## 7. Download reports and save the executed notebook
Download reports below. Separately use **File → Download → Download .ipynb** to save this notebook with its actual outputs. Put the notebook in your GitHub notebooks folder and reports under reports/module1. Keep raw archives out of Git unless the repository policy permits them.

In [ ]:
archive = shutil.make_archive('/content/RouteRanker_Module1_Reports', 'zip', root_dir=str(REPORTS))
files.download(archive)